# 00 — Overview del sistema hospitalario

Este notebook explica la arquitectura completa de **laSalle Health Center**: servicios Docker, almacenamiento, flujo de datos, modelos de IA, dashboard, monitorización y automatización.

Úsalo para abrir la defensa del proyecto y explicar que no es un script aislado, sino una infraestructura containerizada.

## 1. Flujo lógico end-to-end

```text
Formulario / CSV / Radiografía
        ↓
Dashboard / API
        ↓
Pipeline
        ↓
PostgreSQL + MongoDB + MinIO
        ↓
Modelos IA:
  - ml-triage
  - ml-inference
        ↓
Dashboard + informes + alertas + logs
```

La idea principal es separar responsabilidades:

- el dashboard muestra y recoge datos;
- la API expone endpoints;
- el pipeline procesa;
- las bases de datos persisten;
- los modelos predicen;
- Loki/Promtail centralizan logs;
- automation crea alertas.

In [3]:
# Comprobación de carpetas clave
paths = [
    "docker-compose.yml",
    "Dockerfile",
    "services/api",
    "services/dashboard",
    "services/pipeline",
    "services/ml-triage",
    "services/ml-inference",
    "services/automation",
    "monitoring",
    "models",
    "specs",
    "docs/diario-ia",
]

status = []
for p in paths:
    path = ROOT / p
    status.append({"ruta": p, "existe": path.exists(), "tipo": "dir" if path.is_dir() else "file" if path.is_file() else "missing"})

pd.DataFrame(status)


,ruta,existe,tipo
0,docker-compose.yml,True,file
1,Dockerfile,True,file
2,services/api,True,dir
3,services/dashboard,True,dir
4,services/pipeline,True,dir
5,services/ml-triage,True,dir
6,services/ml-inference,True,dir
7,services/automation,True,dir
8,monitoring,True,dir
9,models,True,dir


## 2. Servicios Docker Compose

La arquitectura se levanta con `docker compose up -d`. Cada servicio tiene una responsabilidad separada.

In [4]:
# Lectura de docker-compose.yml para listar servicios.
# Si PyYAML no está disponible, muestra una instrucción alternativa.
compose_path = ROOT / "docker-compose.yml"
try:
    import yaml
    compose = yaml.safe_load(compose_path.read_text(encoding="utf-8"))
    rows = []
    for name, cfg in compose.get("services", {}).items():
        ports = cfg.get("ports", [])
        depends = cfg.get("depends_on", [])
        if isinstance(depends, dict):
            depends = list(depends.keys())
        rows.append({
            "servicio": name,
            "imagen/target": cfg.get("image") or (cfg.get("build") or {}).get("target"),
            "puertos": ", ".join(map(str, ports)) if ports else "interno",
            "depends_on": ", ".join(depends) if depends else "—",
        })
    services_df = pd.DataFrame(rows)
    services_df
except Exception as exc:
    print("No se pudo parsear docker-compose.yml con PyYAML:", exc)
    print("Puedes verlo con: docker compose config --services")


## 3. Capa de almacenamiento

| Tecnología | Uso | Justificación |
|---|---|---|
| PostgreSQL | Pacientes, ingresos y datos estructurados | Integridad relacional, SQL y consistencia. |
| MongoDB | Predicciones, eventos, rechazos, alertas y documentos flexibles | Esquema flexible para outputs de modelos y eventos. |
| MinIO | Landing zone/raw: CSVs, JSONs online, posibles imágenes | Patrón tipo S3 reproducible localmente. |

Esto cumple el requisito de usar más de un tipo de almacenamiento y simula una arquitectura Big Data real.

In [6]:
storage = pd.DataFrame([
    {"capa": "Raw / landing zone", "tecnologia": "MinIO", "ejemplos": "CSV batch, JSON online, imágenes originales"},
    {"capa": "Estructurada", "tecnologia": "PostgreSQL", "ejemplos": "pacientes, ingresos"},
    {"capa": "Documental/eventos", "tecnologia": "MongoDB", "ejemplos": "predictions, system_events, ingestion_rejects, alerts"},
])
storage


,capa,tecnologia,ejemplos
0,Raw / landing zone,MinIO,"CSV batch, JSON online, imágenes originales"
1,Estructurada,PostgreSQL,"pacientes, ingresos"
2,Documental/eventos,MongoDB,"predictions, system_events, ingestion_rejects,..."


## 4. Modelos de IA del proyecto

| Servicio | Modelo | Entrada | Salida |
|---|---|---|---|
| `ml-triage` | Modelo tabular de triaje | Síntomas y antecedentes | Alta / Media / Baja |
| `ml-triage` | Modelo tabular de enfermedad | Síntomas y antecedentes | Sospecha diagnóstica orientativa |
| `ml-inference` | Deep Learning radiografías | Imagen PNG/JPG | Sana / Neumonía / COVID-19 |

Los modelos están versionados en `models/`. Los pesos pesados no deben subirse a Git, pero sí las métricas, matrices y análisis crítico.

## 5. Matirzes de confusión

Orden recomendado:

1. `docker compose ps`: demostrar servicios.
2. Dashboard: pacientes + triaje + radiografías.
3. Pipeline batch: seed + batch + eventos.
4. Calidad: CSV inválido con rechazos.
5. Modelos tabulares: métricas y matrices.
6. Radiografías: comparación CNN / ResNet18 / EfficientNet-B0.
7. Loki/Promtail: logging centralizado.
8. Automation: alertas en MongoDB.
9. Ética/legal: sistema de apoyo, no diagnóstico.